# VLSP 2025 KD — GRPO Fine-tuning

**Goal**: Run GRPO (Group Relative Policy Optimization) on top of the SFT model to improve answer accuracy via RL.

**Prerequisite**: Run either NB2 (SFT-only) or NB3 (KD=Distill+SFT) first, then set `SFT_MODEL_INPUT` below.

| Phase | Time estimate |
|-------|---------------|
| Merge SFT LoRA + GRPO training (1 epoch, 2993 samples) | ~3–5h |
| Greedy eval (584 samples) | ~30–45 min |
| **Total** | **~3.5–5.5h** |

## Execution
1. Set `SFT_MODEL_INPUT` to the path of your SFT adapter from NB2 or NB3
2. `TEST_MODE=True` → quick sanity check (50 samples, ~30 min)
3. `TEST_MODE=False` → full run

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 1: CONFIGURATION
# ════════════════════════════════════════════════════════════════

TEST_MODE    = True      # ← SET TO False AFTER TEST PASSES
TEST_SAMPLES = 50        # samples in test mode

NOTEBOOK_ID = "grpo"
OUTPUT_DIR  = f"/kaggle/working/outputs/{NOTEBOOK_ID}"

# ─── Set path to the SFT adapter from NB2 or NB3 ──────────────
# NB3 (KD) output: '/kaggle/working/outputs/kd/sft_adapter'
# NB2 (SFT-only) output: '/kaggle/working/outputs/sft-only/sft_adapter'
# Kaggle dataset input: '/kaggle/input/datasets/thanhduc1180/vlsp2025-sft-adapter/...'
# Leave None to auto-detect from disk
SFT_MODEL_INPUT = None
# ───────────────────────────────────────────────────────────────

import os
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"{'='*60}")
print(f"Notebook           : {NOTEBOOK_ID}  (GRPO RL fine-tuning)")
print(f"TEST_MODE          : {TEST_MODE}  (n={TEST_SAMPLES if TEST_MODE else 'ALL'})")
print(f"SFT_MODEL_INPUT    : {SFT_MODEL_INPUT or '(auto-detect)'}")
print(f"Output             : {OUTPUT_DIR}")
print(f"{'='*60}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 2: INSTALL WHEELS + COPY PIPELINE CODE
# ════════════════════════════════════════════════════════════════
import subprocess, os, sys, shutil
from pathlib import Path

WORK_DIR     = Path("/kaggle/working/vlsp2025")
WHEELS_DIR   = Path("/kaggle/input/datasets/thanhduc1180/vlsp2025-kd-wheels")
PIPELINE_SRC = Path("/kaggle/input/datasets/thanhduc1180/vlsp2025-kd-pipeline")

os.environ.update({
    "HF_HUB_OFFLINE":          "1",
    "TRANSFORMERS_OFFLINE":     "1",
    "TOKENIZERS_PARALLELISM":   "false",
    "PYTORCH_CUDA_ALLOC_CONF":  "expandable_segments:True",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Install wheels
if WHEELS_DIR.exists():
    wheels = sorted(WHEELS_DIR.glob("*.whl"))
    if wheels:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-index",
             f"--find-links={WHEELS_DIR}"] + [str(w) for w in wheels],
            capture_output=True, text=True)
        print(f"Wheels: {len(wheels)} installed" if r.returncode == 0 else f"Wheel warn: {r.stderr[:200]}")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "transformers", "accelerate", "peft", "bitsandbytes", "trl", "tqdm"],
                   check=False)
    print("Installed from PyPI (online mode)")

# Copy pipeline code
if PIPELINE_SRC.exists():
    for d in ["pipeline", "src", "configs"]:
        src_d = PIPELINE_SRC / d
        dst_d = WORK_DIR / d
        if src_d.exists():
            if dst_d.exists():
                shutil.rmtree(dst_d)
            shutil.copytree(src_d, dst_d)
            n = sum(1 for _ in dst_d.rglob("*") if _.is_file())
            print(f"  Copied {d}/ ({n} files)")
        else:
            print(f"  WARNING: {d}/ not found in pipeline dataset")
    if not (WORK_DIR / "pipeline" / "__init__.py").exists():
        raise RuntimeError("pipeline/__init__.py missing — re-upload vlsp2025-kd-pipeline dataset")
elif (WORK_DIR / "pipeline").exists():
    print(f"Pipeline already present: {WORK_DIR / 'pipeline'}")
else:
    raise RuntimeError(f"Pipeline not found at {PIPELINE_SRC}")

if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)
print(f"WORK_DIR: {WORK_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 3: GPU + LIBRARY CHECK
# ════════════════════════════════════════════════════════════════
import sys, os, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable RTX 6000 accelerator in notebook settings.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU:  {gpu_name}  ({vram_gb:.0f} GB)")

import transformers, peft, accelerate
print(f"Transformers: {transformers.__version__}  PEFT: {peft.__version__}")

try:
    import trl
    print(f"TRL: {trl.__version__}  (GRPOTrainer available)")
    HAS_TRL = True
except ImportError:
    print("TRL: not available — will use manual GRPO")
    HAS_TRL = False

try:
    import flash_attn
    HAS_FLASH = True
    print(f"Flash Attn: {flash_attn.__version__}")
except ImportError:
    HAS_FLASH = False
    print("Flash Attn: not available (sdpa fallback)")

GPU_PROFILE = "rtx6000_96gb" if vram_gb > 80 else ("a100_80gb" if vram_gb > 60 else "p100_16gb")
print(f"GPU Profile: {GPU_PROFILE}")

def resolve_model(hf_id):
    name = hf_id.split("/")[-1].lower()
    for root in [Path("/kaggle/input"), Path("/kaggle/models")]:
        if not root.exists(): continue
        for d in root.rglob("config.json"):
            parent = d.parent
            if name.replace("-","").replace("_","") in parent.name.lower().replace("-","").replace("_",""):
                print(f"  Found offline: {parent}")
                return str(parent)
    return hf_id

STUDENT_PATH = resolve_model("Qwen/Qwen3.5-4B")
if STUDENT_PATH == "Qwen/Qwen3.5-4B":
    p = Path("/kaggle/input/models/thanhduc1180/qwen_35_4b/transformers/default/1")
    if p.exists(): STUDENT_PATH = str(p)
print(f"Student base: {STUDENT_PATH}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 4: CONFIG
# ════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path
from pipeline.config import load_config, save_config

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

cfg = load_config(gpu_profile=GPU_PROFILE, overrides={
    "model": {
        "student_model": STUDENT_PATH,
        "use_flash_attention": HAS_FLASH,
    },
    "data": {
        "vinumqa_train":        "/kaggle/input/datasets/thanhduc1180/vinumericalqa-private/train.json",
        "vinumqa_valid":        "/kaggle/input/datasets/thanhduc1180/vinumericalqa-private/valid.json",
        "vinumqa_test":         "/kaggle/input/datasets/thanhduc1180/vinumericalqa-private/test.json",
        "vinumqa_private_test": "/kaggle/input/datasets/thanhduc1180/vinumericalqa-private/private_test.json",
        "finqa_dir":            "/kaggle/input/datasets/thanhduc1180/finqa-en",
        "max_samples":          TEST_SAMPLES if TEST_MODE else None,
    },
    "grpo": {
        "num_epochs": 1,
        "lora_r": 32,
        "num_generations": 5,
        "max_completion_length": 512,
    },
    "inference": {"num_candidates": 1, "batch_size": 8},
})

print(f"Student base : {cfg.model.student_model.split('/')[-1]}")
print(f"GRPO: epochs={cfg.grpo.num_epochs}  LoRA r={cfg.grpo.lora_r}  "
      f"num_generations={cfg.grpo.num_generations}  max_completion={cfg.grpo.max_completion_length}")
save_config(cfg, str(WORK_DIR / "data/pipeline/config_grpo.yaml"))

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 5: DATA PREP (load or generate GRPO parquet)
# ════════════════════════════════════════════════════════════════
import json, time, os, sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

pipeline_out = WORK_DIR / "data/pipeline"
grpo_train_path = pipeline_out / "grpo_train.parquet"
grpo_valid_path = pipeline_out / "grpo_valid.parquet"

if grpo_train_path.exists() and grpo_valid_path.exists():
    import pandas as pd
    train_df = pd.read_parquet(grpo_train_path)
    valid_df  = pd.read_parquet(grpo_valid_path)
    if TEST_MODE:
        train_df = train_df.head(TEST_SAMPLES)
        valid_df  = valid_df.head(TEST_SAMPLES)
    print(f"Loaded existing GRPO data: train={len(train_df)}, valid={len(valid_df)}")
else:
    print("GRPO parquet not found — running data prep...")
    from pipeline.data_prep import run_data_prep
    t0 = time.time()
    data_paths = run_data_prep(cfg)
    print(f"Data prep done in {time.time()-t0:.1f}s")
    import pandas as pd
    train_df = pd.read_parquet(grpo_train_path)
    valid_df  = pd.read_parquet(grpo_valid_path)
    if TEST_MODE:
        train_df = train_df.head(TEST_SAMPLES)
        valid_df  = valid_df.head(TEST_SAMPLES)

print(f"GRPO train: {len(train_df)} samples  |  valid: {len(valid_df)} samples")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 6: RESOLVE SFT MODEL (auto-detect from disk)
# Priority: explicit path > NB3 kd output > NB2 sft-only output >
#           Kaggle dataset input > base student model
# ════════════════════════════════════════════════════════════════
import sys, os
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

def _find_sft_model():
    # 1. Explicit config
    if SFT_MODEL_INPUT and Path(SFT_MODEL_INPUT).exists():
        return str(SFT_MODEL_INPUT), "explicit config"

    # 2. NB3 output (KD pipeline)
    for p in [
        Path("/kaggle/working/outputs/kd/sft_adapter"),
        Path("/kaggle/working/vlsp2025/checkpoints/sft_kd/final"),
    ]:
        if (p / "adapter_config.json").exists():
            return str(p), "NB3 KD output"

    # 3. NB2 output (SFT-only)
    for p in [
        Path("/kaggle/working/outputs/sft-only/sft_adapter"),
        Path("/kaggle/working/vlsp2025/checkpoints/sft/final"),
    ]:
        if (p / "adapter_config.json").exists():
            return str(p), "NB2 SFT-only output"

    # 4. Kaggle dataset with uploaded SFT adapter
    for root in [Path("/kaggle/input")]:
        if not root.exists(): continue
        for p in root.rglob("adapter_config.json"):
            return str(p.parent), f"Kaggle dataset: {p.parent}"

    # 5. Fallback to base model (no SFT)
    return globals().get("STUDENT_PATH", "Qwen/Qwen3.5-4B"), "base model (no SFT)"

sft_model_path, sft_source = _find_sft_model()
is_lora = (Path(sft_model_path) / "adapter_config.json").exists()

print(f"SFT model path : {sft_model_path}")
print(f"Source         : {sft_source}")
print(f"Is LoRA adapter: {is_lora}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 7: MERGE SFT LoRA (if adapter) → ready for GRPO
# Merging avoids double-LoRA complexity and reduces inference memory.
# Time: ~2–3 min
# ════════════════════════════════════════════════════════════════
import gc, sys, os, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)

MERGED_DIR = Path("/kaggle/working/outputs/grpo/sft_merged")

# Determine dtype key for this transformers version
import inspect as _insp
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from transformers import AutoModelForCausalLM as _ACM, AutoTokenizer
_dtype_key = "dtype" if "dtype" in _insp.signature(_ACM.from_pretrained).parameters else "torch_dtype"
dtype = torch.bfloat16 if cfg.sft.bf16 else torch.float16

if is_lora:
    if MERGED_DIR.exists() and (MERGED_DIR / "config.json").exists():
        print(f"Merged model already exists: {MERGED_DIR}")
        grpo_base_path = str(MERGED_DIR)
    else:
        print(f"Merging LoRA adapter: {sft_model_path}")
        print(f"  Base model: {cfg.model.student_model}")

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        from pipeline import _load_model_robust, _load_tokenizer_robust
        from peft import PeftModel

        base_model = _load_model_robust(
            cfg.model.student_model,
            {"trust_remote_code": True, _dtype_key: dtype, "device_map": "auto", "low_cpu_mem_usage": True}
        )
        peft_model = PeftModel.from_pretrained(base_model, sft_model_path)
        print("  Merging weights...")
        merged = peft_model.merge_and_unload()

        MERGED_DIR.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(str(MERGED_DIR))

        tok = _load_tokenizer_robust(sft_model_path, trust_remote_code=True)
        tok.save_pretrained(str(MERGED_DIR))

        del merged, peft_model, base_model
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        grpo_base_path = str(MERGED_DIR)
        sz = sum(p.stat().st_size for p in MERGED_DIR.rglob("*") if p.is_file()) / 1024**3
        print(f"  Merged model saved: {MERGED_DIR}  ({sz:.1f} GB)")
        free_gb = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"  GPU free: {free_gb:.1f} GB")
else:
    grpo_base_path = sft_model_path
    print(f"Using full model for GRPO: {grpo_base_path}")

print(f"\nGRPO will start from: {grpo_base_path}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 8: GRPO TRAINING
# Uses TRL GRPOTrainer with PCPO reward: R = R_valid*(0.7 + 0.2*R_exec + 0.1*R_bonus)
# Time: ~3–5h (2993 samples, 1 epoch, 5 generations per sample)
# ════════════════════════════════════════════════════════════════
import gc, os, sys, time, torch, warnings
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))
os.chdir(WORK_DIR)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, message=".*warmup_ratio.*")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free before GRPO: {free_gb:.1f} GB")

print("="*60)
print("GRPO TRAINING")
print("="*60)
print(f"  Base model   : {grpo_base_path.split('/')[-1] or grpo_base_path}")
print(f"  Train samples: {len(train_df)}")
print(f"  Num epochs   : {cfg.grpo.num_epochs}")
print(f"  Num gens/step: {cfg.grpo.num_generations}")
print(f"  Max completion: {cfg.grpo.max_completion_length} tokens")
print(f"  LoRA r={cfg.grpo.lora_r}  bf16={cfg.grpo.bf16}")

from pipeline.train_grpo import run_grpo_training
from pipeline.reward import compute_pcpo_reward

t0 = time.time()

try:
    grpo_model_path = run_grpo_training(cfg, sft_model_path=grpo_base_path)
except Exception as e:
    print(f"\nGRPO training failed: {type(e).__name__}: {e}")
    print("Falling back to SFT model for evaluation...")
    grpo_model_path = grpo_base_path

elapsed = time.time() - t0
print(f"\nGRPO done in {elapsed/3600:.2f}h ({elapsed:.0f}s)")
print(f"Model: {grpo_model_path}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free after GRPO: {free_gb:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 9: GREEDY EVALUATION
# Single-pass greedy inference (no majority voting)
# Time: ~30–45 min for 584 valid samples
# ════════════════════════════════════════════════════════════════
import gc, json, time, torch, re, warnings
from pathlib import Path
from tqdm import tqdm
from pipeline import _load_model_robust, _load_tokenizer_robust
from pipeline.evaluate import answers_match, programs_match
from pipeline.program_executor import validate_program

warnings.filterwarnings("ignore", category=FutureWarning)

WORK_DIR = Path("/kaggle/working/vlsp2025")
VALID_PATH = str(WORK_DIR / "data/pipeline/sft_valid.json")
if not Path(VALID_PATH).exists():
    VALID_PATH = "/kaggle/input/datasets/thanhduc1180/vinumericalqa-private/valid.json"

with open(VALID_PATH, "r", encoding="utf-8") as f:
    valid_data = json.load(f)

if TEST_MODE:
    valid_data = valid_data[:min(TEST_SAMPLES, len(valid_data))]
    print(f"TEST MODE: using {len(valid_data)} validation samples")

# Resolve display name
_model_display = grpo_model_path
for _part in reversed(grpo_model_path.replace("\\", "/").split("/")):
    if len(_part) > 3 and _part not in ("final", "default", "1"):
        _model_display = _part; break
print(f"Evaluating {len(valid_data)} samples | Model: {_model_display}")

# Load GRPO model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

import inspect as _insp
from transformers import AutoModelForCausalLM as _ACM
_dtype_key = "dtype" if "dtype" in _insp.signature(_ACM.from_pretrained).parameters else "torch_dtype"
dtype = torch.bfloat16 if cfg.grpo.bf16 else torch.float16
mkw = {"trust_remote_code": True, _dtype_key: dtype, "device_map": "auto", "low_cpu_mem_usage": True}
if HAS_FLASH:
    try:
        import flash_attn; mkw["attn_implementation"] = "flash_attention_2"
    except ImportError:
        mkw["attn_implementation"] = "sdpa"

tokenizer = _load_tokenizer_robust(grpo_model_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load as PEFT or full model
is_grpo_lora = (Path(grpo_model_path) / "adapter_config.json").exists()
if is_grpo_lora:
    from peft import PeftModel
    base = _load_model_robust(grpo_base_path, mkw)
    model = PeftModel.from_pretrained(base, grpo_model_path)
    print("Loaded GRPO model as PEFT (LoRA)")
else:
    model = _load_model_robust(grpo_model_path, mkw)
    print("Loaded GRPO model as full model")

model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
free_gb = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Model loaded. GPU free: {free_gb:.1f} GB")

def _extract(text):
    pm = list(re.finditer(r"\*\*Chương trình tính toán:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text))
    am = list(re.finditer(r"\*\*Đáp án cuối cùng:\*\*\s*((?:.|\n)*?)(?=\s*\*\*|$)", text))
    prog = pm[-1].group(1).strip() if pm else None
    ans  = am[-1].group(1).strip() if am else None
    return prog, ans

BATCH_SIZE = 8
MAX_NEW_TOKENS = 512
eval_results = []
batches = [valid_data[i:i+BATCH_SIZE] for i in range(0, len(valid_data), BATCH_SIZE)]
t0 = time.time()

for b_idx, batch in enumerate(tqdm(batches, desc="Greedy eval")):
    prompts   = [s["messages"][0]["content"] if "messages" in s else s.get("question", "") for s in batch]
    gold_prog = [s.get("metadata", {}).get("program", s.get("program", "")) for s in batch]
    gold_ans  = [str(s.get("metadata", {}).get("answer", s.get("answer", ""))) for s in batch]

    texts_in = []
    for p in prompts:
        msgs = [{"role": "user", "content": p}]
        try:
            t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        texts_in.append(t)

    enc = tokenizer(texts_in, return_tensors="pt", truncation=True, max_length=3584, padding=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    in_len = enc["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.batch_decode(out[:, in_len:], skip_special_tokens=True)

    for i, (txt, samp) in enumerate(zip(gen, batch)):
        pp, pa = _extract(txt)
        ea = answers_match(pa or "", gold_ans[i])
        pa_m = programs_match(pp or "", gold_prog[i])
        eval_results.append({
            "id": samp.get("id", f"s{len(eval_results)}"),
            "pred_program": pp or "",
            "pred_answer":  pa or "",
            "gold_program": gold_prog[i],
            "gold_answer":  gold_ans[i],
            "ea": ea, "pa": pa_m,
            "valid": validate_program(pp) if pp else False,
            "raw_output": txt[:300],
        })

    if (b_idx + 1) % 10 == 0:
        el = time.time() - t0
        eta = (len(batches) - b_idx - 1) / ((b_idx + 1) / el)
        n_done = min((b_idx + 1) * BATCH_SIZE, len(valid_data))
        ea_now = sum(r["ea"] for r in eval_results) / len(eval_results) * 100
        print(f"  [{n_done}/{len(valid_data)}] EA={ea_now:.1f}%  ETA={eta/60:.0f}min")

elapsed = time.time() - t0
N  = len(eval_results)
EA = sum(r["ea"] for r in eval_results) / N * 100 if N else 0
PA = sum(r["pa"] for r in eval_results) / N * 100 if N else 0
VR = sum(r["valid"] for r in eval_results) / N * 100 if N else 0

print(f"\n{'='*60}")
print(f"GREEDY EVALUATION ({N} samples, {elapsed:.0f}s / {elapsed/60:.1f}min)")
print(f"{'='*60}")
print(f"  Execution Accuracy (EA): {EA:.2f}%   ({sum(r['ea'] for r in eval_results)}/{N})")
print(f"  Program Accuracy   (PA): {PA:.2f}%   ({sum(r['pa'] for r in eval_results)}/{N})")
print(f"  Valid Program Rate:      {VR:.2f}%")
print(f"{'='*60}")

eval_summary = {
    "execution_accuracy": EA / 100,
    "program_accuracy":   PA / 100,
    "valid_rate":         VR / 100,
    "total": N,
    "ea_correct": sum(r["ea"] for r in eval_results),
    "pa_correct": sum(r["pa"] for r in eval_results),
    "details": eval_results,
}

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    print(f"GPU free after eval: {free_gb:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 10: COMPARISON WITH BASELINES
# ════════════════════════════════════════════════════════════════
import json
from pathlib import Path

grpo_ea = eval_summary["execution_accuracy"]
grpo_pa = eval_summary["program_accuracy"]
grpo_vr = eval_summary["valid_rate"]

baselines = {}
for nb_id, nb_label in [("sft-only", "SFT-only (NB2)"), ("kd", "KD/Distill+SFT (NB3)")]:
    p = Path(f"/kaggle/working/outputs/{nb_id}/eval_results.json")
    if p.exists():
        baselines[nb_label] = json.load(open(p))

print(f"\n{'='*70}")
print(f"COMPARISON ACROSS NOTEBOOKS")
print(f"{'='*70}")
print(f"{'Model':<40} {'EA':>8} {'PA':>8} {'Valid%':>8}")
print("-"*68)
for label, r in baselines.items():
    print(f"{label:<40} {r['execution_accuracy']:>7.2%} {r['program_accuracy']:>7.2%} {r['valid_rate']:>7.2%}")
print(f"{'GRPO (this notebook)':<40} {grpo_ea:>7.2%} {grpo_pa:>7.2%} {grpo_vr:>7.2%}")
print("="*70)

for label, r in baselines.items():
    delta = grpo_ea - r["execution_accuracy"]
    sign = "+" if delta >= 0 else ""
    print(f"  GRPO vs {label}: EA {sign}{delta:.2%}")

if not baselines:
    print(f"No baseline results found. Run NB2/NB3 first for comparison.")
    print(f"\nGRPO results: EA={grpo_ea:.2%}  PA={grpo_pa:.2%}  Valid={grpo_vr:.2%}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# CELL 11: SAVE RESULTS
# ════════════════════════════════════════════════════════════════
import gc, json, shutil, torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working/vlsp2025")
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

# Save eval results
eval_out = dict(notebook=NOTEBOOK_ID, test_mode=TEST_MODE, model_path=grpo_model_path,
                sft_source=sft_source)
for k, v in eval_summary.items():
    if k != "details": eval_out[k] = v
eval_out["details"] = eval_summary["details"]
with open(out / "eval_results.json", "w", encoding="utf-8") as f:
    json.dump(eval_out, f, ensure_ascii=False, indent=2)
print(f"eval_results.json saved  (EA={eval_out['execution_accuracy']:.2%}  PA={eval_out['program_accuracy']:.2%})")

# Compact predictions
preds = [
    dict(id=r["id"], pred_answer=r["pred_answer"], gold_answer=r["gold_answer"], ea=r["ea"], pa=r["pa"])
    for r in eval_summary["details"]
]
with open(out / "predictions.json", "w", encoding="utf-8") as f:
    json.dump(preds, f, ensure_ascii=False, indent=2)

# Copy GRPO model adapter (LoRA weights only, ~1 GB)
model_save = out / "grpo_adapter"
if model_save.exists(): shutil.rmtree(model_save)
if Path(grpo_model_path).exists():
    shutil.copytree(grpo_model_path, model_save)
    sz = sum(p.stat().st_size for p in model_save.rglob("*") if p.is_file()) / 1024**2
    print(f"GRPO adapter saved -> {model_save}  ({sz:.0f} MB)")

# Copy merged SFT model if present (useful for next session)
if MERGED_DIR.exists():
    merged_save = out / "sft_merged_snapshot"
    if not merged_save.exists():
        print(f"(Skipping merged SFT copy — {MERGED_DIR} would take ~8 GB disk)")
        print(f"  To export: shutil.copytree(str(MERGED_DIR), str(out/'sft_merged'))")

print(f"\nAll outputs -> {OUTPUT_DIR}")
print(f"Files: {sorted(p.name for p in out.iterdir())}")
print(f"\nFINAL SUMMARY  ({'TEST MODE' if TEST_MODE else 'FULL RUN'}):")
print(f"  EA (Execution Accuracy): {eval_summary['execution_accuracy']:.2%}")
print(f"  PA (Program Accuracy)  : {eval_summary['program_accuracy']:.2%}")
print(f"  Valid Program Rate     : {eval_summary['valid_rate']:.2%}")
if TEST_MODE:
    print(f"\n  TEST_MODE=True — set to False and restart kernel for real results.")